In [1]:

# IMPORT LIBRARY

import requests
import pandas as pd
import json
import os
import hashlib
import time
from datetime import datetime

print("Semua library berhasil dipanggil!")


Semua library berhasil dipanggil!


In [2]:

# MEMBUAT STRUKTUR FOLDER PROYEK

folder_proyek = "rtm1_2501010166"

os.makedirs(f"{folder_proyek}/notebooks", exist_ok=True)
os.makedirs(f"{folder_proyek}/data/raw", exist_ok=True)
os.makedirs(f"{folder_proyek}/docs", exist_ok=True)

print("Folder proyek berhasil dibuat!")
print("\nStruktur folder:")

for root, dirs, files in os.walk(folder_proyek):
    level = root.replace(folder_proyek, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")


Folder proyek berhasil dibuat!

Struktur folder:
rtm1_2501010166/
    data/
        raw/
    notebooks/
    docs/


In [4]:

# Menghubungkan ke API OpenStreetMap

url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:90];

(
  nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet|bed_and_breakfast)$"]
  (-8.85,114.4,-8.0,115.75);
);

out center tags;
"""

headers = {
    "User-Agent": "AkuisisiHotelBaliINSTIKI/1.0 (email: EMAIL_KAMPUS_KAMU)"
}

try:
    response = requests.post(
        url,
        data={"data": query},
        headers=headers,
        timeout=120
    )

    response.raise_for_status()
    data = response.json()

    print("Berhasil terhubung ke API OpenStreetMap!")
    print("Jumlah data ditemukan:", len(data["elements"]))

    # Menampilkan satu contoh data
    if len(data["elements"]) > 0:
        print("\nContoh data pertama:")
        print(data["elements"][0])

except requests.exceptions.RequestException as e:
    print("Gagal mengambil data:", e)


Berhasil terhubung ke API OpenStreetMap!
Jumlah data ditemukan: 6122

Contoh data pertama:
{'type': 'node', 'id': 260947786, 'lat': -8.3385652, 'lon': 115.6598117, 'tags': {'name': 'Coral View Cafe', 'tourism': 'chalet'}}


In [6]:
# MENYIMPAN DATA MENTAH JSON

nama_file = "hotel_bali.json"

path_file = f"{folder_proyek}/data/raw/{nama_file}"

with open(path_file, "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

print("Data JSON berhasil disimpan!")
print("Nama file:", nama_file)
print("Jumlah objek:", len(data["elements"]))

Data JSON berhasil disimpan!
Nama file: hotel_bali.json
Jumlah objek: 6122


In [10]:
# MENGUBAH DATA JSON MENJADI CSV

rows = []

for item in data["elements"]:
    tags = item.get("tags", {})
    center = item.get("center", {})

    latitude = item.get("lat", center.get("lat"))
    longitude = item.get("lon", center.get("lon"))

    rows.append({
        "osm_type": item.get("type"),
        "osm_id": item.get("id"),
        "name": tags.get("name"),
        "tourism": tags.get("tourism"),
        "latitude": latitude,
        "longitude": longitude
    })

df_hotel = pd.DataFrame(rows)

path_csv = f"{folder_proyek}/data/raw/hotel_bali.csv"

df_hotel.to_csv(path_csv, index=False, encoding="utf-8-sig")

print("Data berhasil diubah menjadi CSV!")
print("Jumlah baris dan kolom:", df_hotel.shape)

Data berhasil diubah menjadi CSV!
Jumlah baris dan kolom: (6122, 6)


In [8]:
# MEMERIKSA DATA CSV

print("Nama file: hotel_bali.csv")
print("Jumlah data:", len(df_hotel))
print("Jumlah kolom:", len(df_hotel.columns))

print("\nNama kolom:")
print(df_hotel.columns.tolist())

print("\nContoh 5 data pertama:")
display(df_hotel.head())

Nama file: hotel_bali.csv
Jumlah data: 6122
Jumlah kolom: 6

Nama kolom:
['osm_type', 'osm_id', 'name', 'tourism', 'latitude', 'longitude']

Contoh 5 data pertama:


,osm_type,osm_id,name,tourism,latitude,longitude
0,node,260947786,Coral View Cafe,chalet,-8.338565,115.659812
1,node,316954328,Topi Inn,guest_house,-8.530209,115.511776
2,node,316955317,Puri Rai,hotel,-8.530337,115.509925
3,node,316956081,Pondok Wisata Serangan 2,guest_house,-8.529722,115.509134
4,node,316959068,Zen Inn,guest_house,-8.532257,115.507215


In [12]:
# MEMERIKSA KUALITAS DATA

print("Jumlah data kosong setiap kolom:")
print(df_hotel.isnull().sum())

print("\nJumlah data duplikat:")
print(df_hotel.duplicated().sum())

Jumlah data kosong setiap kolom:
osm_type       0
osm_id         0
name         825
tourism        0
latitude       0
longitude      0
dtype: int64

Jumlah data duplikat:
0


In [13]:
# MEMBUAT DOKUMENTASI PROVENANCE

path_csv = f"{folder_proyek}/data/raw/hotel_bali.csv"

# Menghitung ukuran file
ukuran_file = os.path.getsize(path_csv)

# Menghitung MD5
md5_hash = hashlib.md5()

with open(path_csv, "rb") as file:
    md5_hash.update(file.read())

checksum = md5_hash.hexdigest()

# Membuat metadata
provenance = {
    "nama_berkas": "hotel_bali.csv",
    "jumlah_data": len(df_hotel),
    "jumlah_kolom": len(df_hotel.columns),
    "nama_kolom": df_hotel.columns.tolist(),
    "ukuran_bytes": ukuran_file,
    "md5": checksum,
    "sumber_data": "OpenStreetMap",
    "metode_akuisisi": "Overpass API",
    "lisensi": "Open Database License (ODbL)",
    "waktu_akuisisi": datetime.now().strftime("%d/%m/%Y %H:%M"),
    "pengambil": {
        "nama": "I Nyoman Cahyanta Kusumadinata",
        "nim": "2501010166"
    }
}

# Menyimpan provenance.json
path_provenance = f"{folder_proyek}/docs/provenance.json"

with open(path_provenance, "w", encoding="utf-8") as file:
    json.dump(provenance, file, ensure_ascii=False, indent=4)

print("File provenance.json berhasil dibuat!")
print(json.dumps(provenance, indent=4, ensure_ascii=False))

File provenance.json berhasil dibuat!
{
    "nama_berkas": "hotel_bali.csv",
    "jumlah_data": 6122,
    "jumlah_kolom": 6,
    "nama_kolom": [
        "osm_type",
        "osm_id",
        "name",
        "tourism",
        "latitude",
        "longitude"
    ],
    "ukuran_bytes": 381371,
    "md5": "335360e9fd879491c3bc6eec1d04cb19",
    "sumber_data": "OpenStreetMap",
    "metode_akuisisi": "Overpass API",
    "lisensi": "Open Database License (ODbL)",
    "waktu_akuisisi": "17/11/2026 12:15",
    "pengambil": {
        "nama": "I Nyoman Cahyanta Kusumadinata",
        "nim": "2501010166"
    }
}


In [14]:
# MEMBUAT DATA ETHICS CHECKLIST

checklist = [
    ["Aspek", "Pemeriksaan", "Status", "Keterangan"],
    ["Sumber data", "Menggunakan sumber data terbuka", "Ya", "OpenStreetMap"],
    ["Lisensi", "Memperhatikan lisensi data", "Ya", "Open Database License (ODbL)"],
    ["Privasi", "Tidak mengambil data pribadi", "Ya", "Data berupa informasi lokasi penginapan"],
    ["Izin akses", "Menggunakan API resmi", "Ya", "Overpass API"],
    ["Keamanan", "Tidak mengambil informasi sensitif", "Ya", "Tidak ada data sensitif"],
    ["Atribusi", "Mencantumkan sumber data", "Ya", "OpenStreetMap"],
    ["Penyimpanan", "Menyimpan data secara bertanggung jawab", "Ya", "Disimpan dalam folder proyek"],
    ["Penggunaan", "Menggunakan data untuk keperluan akademik", "Ya", "Tugas perkuliahan"]
]

df_checklist = pd.DataFrame(
    checklist[1:],
    columns=checklist[0]
)

path_checklist = f"{folder_proyek}/docs/data_ethics_checklist.csv"

df_checklist.to_csv(
    path_checklist,
    index=False,
    encoding="utf-8-sig"
)

print("File data_ethics_checklist.csv berhasil dibuat!")
display(df_checklist)

File data_ethics_checklist.csv berhasil dibuat!


,Aspek,Pemeriksaan,Status,Keterangan
0,Sumber data,Menggunakan sumber data terbuka,Ya,OpenStreetMap
1,Lisensi,Memperhatikan lisensi data,Ya,Open Database License (ODbL)
2,Privasi,Tidak mengambil data pribadi,Ya,Data berupa informasi lokasi penginapan
3,Izin akses,Menggunakan API resmi,Ya,Overpass API
4,Keamanan,Tidak mengambil informasi sensitif,Ya,Tidak ada data sensitif
5,Atribusi,Mencantumkan sumber data,Ya,OpenStreetMap
6,Penyimpanan,Menyimpan data secara bertanggung jawab,Ya,Disimpan dalam folder proyek
7,Penggunaan,Menggunakan data untuk keperluan akademik,Ya,Tugas perkuliahan


In [15]:
# MEMBUAT README.md

readme = """
# Akuisisi Data Hotel dan Penginapan di Bali

## 1. Deskripsi
Proyek ini merupakan tugas akuisisi data yang mengambil informasi hotel dan penginapan di Bali menggunakan OpenStreetMap melalui Overpass API.

## 2. Sumber Data
- Sumber: OpenStreetMap
- Metode: Overpass API
- Lisensi: Open Database License (ODbL)

## 3. Informasi Dataset
- Nama file: hotel_bali.csv
- Jumlah data: 6122 baris
- Jumlah kolom: 6
- Kolom: osm_type, osm_id, name, tourism, latitude, longitude

## 4. Struktur Folder
- notebooks: menyimpan notebook Google Colab
- data/raw: menyimpan data mentah JSON dan hasil CSV
- docs: menyimpan dokumentasi provenance dan data ethics checklist

## 5. Dokumentasi
- provenance.json: informasi sumber, ukuran, checksum, dan waktu akuisisi data
- data_ethics_checklist.csv: pemeriksaan etika penggunaan data

## 6. Identitas Pengambil Data
- Nama: I Nyoman Cahyanta Kusumadinata
- NIM: 2501010166

## 7. Lisensi dan Atribusi
Data bersumber dari OpenStreetMap dan digunakan dengan memperhatikan ketentuan Open Database License (ODbL).
"""

path_readme = f"{folder_proyek}/README.md"

with open(path_readme, "w", encoding="utf-8") as file:
    file.write(readme)

print("File README.md berhasil dibuat!")
print(readme)

File README.md berhasil dibuat!

# Akuisisi Data Hotel dan Penginapan di Bali

## 1. Deskripsi
Proyek ini merupakan tugas akuisisi data yang mengambil informasi hotel dan penginapan di Bali menggunakan OpenStreetMap melalui Overpass API.

## 2. Sumber Data
- Sumber: OpenStreetMap
- Metode: Overpass API
- Lisensi: Open Database License (ODbL)

## 3. Informasi Dataset
- Nama file: hotel_bali.csv
- Jumlah data: 6122 baris
- Jumlah kolom: 6
- Kolom: osm_type, osm_id, name, tourism, latitude, longitude

## 4. Struktur Folder
- notebooks: menyimpan notebook Google Colab
- data/raw: menyimpan data mentah JSON dan hasil CSV
- docs: menyimpan dokumentasi provenance dan data ethics checklist

## 5. Dokumentasi
- provenance.json: informasi sumber, ukuran, checksum, dan waktu akuisisi data
- data_ethics_checklist.csv: pemeriksaan etika penggunaan data

## 6. Identitas Pengambil Data
- Nama: I Nyoman Cahyanta Kusumadinata
- NIM: 2501010166

## 7. Lisensi dan Atribusi
Data bersumber dari OpenStreetM

In [16]:
# MEMERIKSA STRUKTUR FOLDER AKHIR

print("STRUKTUR FOLDER PROYEK\n")

for root, dirs, files in os.walk(folder_proyek):
    level = root.replace(folder_proyek, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = "    " * (level + 1)

    for file in files:
        print(f"{sub_indent}{file}")

STRUKTUR FOLDER PROYEK

rtm1_2501010166/
    README.md
    data/
        raw/
            hotel_bali.json
            hotel_bali.csv
    notebooks/
    docs/
        data_ethics_checklist.csv
        provenance.json
